# 极坐标 $P$-prior：函数编码器训练

## Goal

本 Notebook 用于训练一个极坐标 $P$-prior 条件的函数编码器（FE）。它直接调用项目中的 `train_fe(config, resume=True)`，不会复制或改写训练循环。

- `p_raw`：$c_{\sigma_r}=1$。
- `p_rms`：$c_{\sigma_r}=1/\sqrt{1+4\sigma_r^2}$。
- 两个条件必须分别计算 normalizer、分别从头训练，不能复用权重。
- 正式配置为 FE 500,000 步、`seed=0`、有效 batch size 384。

> 请将本 Notebook 放在 `polar_annulus_p_prior_ablation` 项目根目录，并选择服务器上已正确安装 GPU 版 JAX 的 kernel。不要为了运行 Notebook 重新安装普通 CPU 版 `jax`。

## Setup

### 1. 定位项目并设置 JAX 环境变量

这一单元必须在首次导入 JAX 之前运行。若服务器有多张 GPU，可在导入 JAX 前取消 `CUDA_VISIBLE_DEVICES` 那一行的注释并选择设备。

In [1]:
import os
import sys
from pathlib import Path

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ["CUDA_VISIBLE_DEVICES"] = "4"  # 多 GPU 服务器按需启用

PROJECT_ROOT = Path.cwd().resolve()
REQUIRED_FILES = (
    "config_polar.py",
    "data_polar.py",
    "models_polar.py",
    "train_polar.py",
    "run_prior_ablation.py",
)
missing = [name for name in REQUIRED_FILES if not (PROJECT_ROOT / name).exists()]
if missing:
    raise FileNotFoundError(
        "请把 Notebook 放在项目根目录后重新打开。缺少文件："
        + ", ".join(missing)
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python executable:", sys.executable)

Project root: /home/user/data/Hollon/海洋工程水动力/polar_annulus_p_prior_ablation
Python executable: /home/user/anaconda3/envs/sno/bin/python


### 2. 检查 JAX 和加速器

完整 500k FE 训练不应在 CPU 上启动。这里只接受 JAX 检测到 GPU 或 TPU 后继续。若只想检查 Notebook 结构，可暂时把 `REQUIRE_ACCELERATOR` 改成 `False`，但不要由此启动正式训练。

In [2]:
import jax

REQUIRE_ACCELERATOR = True
devices = jax.devices()
accelerators = [device for device in devices if device.platform in {"gpu", "tpu"}]

print("JAX version:", jax.__version__)
print("JAX devices:", devices)

if REQUIRE_ACCELERATOR and not accelerators:
    raise RuntimeError(
        "JAX 没有检测到 GPU/TPU。请先修复服务器的 CUDA/JAX 环境，"
        "不要在 CPU 上启动 500k FE 正式训练。"
    )

JAX version: 0.9.2
JAX devices: [CudaDevice(id=0)]


## Steps

### 3. 选择当前训练条件

先把 `VARIANT` 设置为 `p_raw`。完成后重启 kernel，再改为 `p_rms`，从头执行本 Notebook。不要同时在同一张 GPU 上训练两个条件。

In [3]:
from config_polar import make_ablation_config

VARIANT = "p_raw"  # 可选："p_raw" 或 "p_rms"
SEED = 0

config = make_ablation_config(
    variant=VARIANT,
    seed=SEED,
    out_dir=PROJECT_ROOT / "out_p_prior_ablation",
)
config.save_json()

print("Variant:", VARIANT)
print("Output directory:", config.output_dir)
print("Config fingerprint:", config.fingerprint())
print("Pressure scaling:", config.pressure_prior_scaling)
print("FE steps:", config.fe_steps)
print("Effective batch size:", config.effective_batch_size)
print("Prior scale pairs:", config.prior_scale_pairs)

assert VARIANT in {"p_raw", "p_rms"}
assert SEED == 0
assert config.fe_steps == 300_000
assert config.effective_batch_size == 768
assert config.prior_scale_pairs == ((0.5, 0.5), (1.0, 1.0), (1.5, 2.0))

Variant: p_raw
Output directory: /home/user/data/Hollon/海洋工程水动力/polar_annulus_p_prior_ablation/out_p_prior_ablation/polar_p_raw_seed0
Config fingerprint: c907573a6dad134e7ae39d145c0f5b7ad041b399cd180f581fd32994ffc9dc8e
Pressure scaling: raw
FE steps: 300000
Effective batch size: 768
Prior scale pairs: ((0.5, 0.5), (1.0, 1.0), (1.5, 2.0))


### 4. 执行正式训练前检查

Preflight 会检查外边界、解析导数、PDE 源项、内边界法向符号、RMS 校准、FE/OL 有限值以及 checkpoint 恢复一致性。任一检查失败都不应开始正式训练。

In [ ]:
from run_prior_ablation import run_preflight_checks

preflight_report = run_preflight_checks(config)
print("Preflight passed:", preflight_report["passed"])
display(preflight_report["checks"])

assert preflight_report["passed"], (
    "Preflight 未通过，请检查："
    + str(config.output_dir / "preflight_report.json")
)

### 5. 启动函数编码器训练

下面的单元会占用当前 kernel，直到 FE 达到 500,000 步。`resume=True` 在没有 checkpoint 时自动从头开始；存在匹配的 checkpoint 时恢复参数、优化器、步数以及训练/评估 PRNG 流。

每 10,000 步覆盖保存 `fe_checkpoint_latest.msgpack`。如果训练在两个 checkpoint 之间中断，最多会丢失最近一次 checkpoint 之后的步数。

训练期间可在另一个 SSH 终端执行：

```bash
tail -f out_p_prior_ablation/polar_p_raw_seed0/fe_training_history.csv
```

In [ ]:
from train_polar import train_fe

fe_state, normalizer = train_fe(
    config,
    resume=True,
)

## Checks

### 6. 确认 FE 完整结束

只有 `completed_steps == 500000` 且配置指纹一致，才把该条件记为 FE 训练完成。

In [ ]:
import json

metadata_path = config.output_dir / "fe_checkpoint_latest.json"
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
display(metadata)

assert metadata["completed_steps"] == config.fe_steps
assert metadata["config_fingerprint"] == config.fingerprint()
assert (config.output_dir / "fe_params.msgpack").exists()

print(f"{VARIANT} FE training completed successfully.")

### 7. 绘制学习曲线

图中只展示已写入训练历史的观测点，不对缺失步数做插值。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

history = np.load(config.output_dir / "fe_training_history.npz")
steps = history["step"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

axes[0].semilogy(steps, history["loss"], label="total")
axes[0].semilogy(steps, history["loss_p"], label="P")
axes[0].semilogy(steps, history["loss_f"], label="f")
axes[0].set(xlabel="FE step", ylabel="Loss", title=f"{VARIANT}: training loss")
axes[0].grid(alpha=0.25)
axes[0].legend()

axes[1].semilogy(steps, history["eval_p_relative_l2"], label="P relative L2")
axes[1].semilogy(steps, history["eval_f_relative_l2"], label="f relative L2")
axes[1].set(xlabel="FE step", ylabel="Evaluation error", title=f"{VARIANT}: held-out error")
axes[1].grid(alpha=0.25)
axes[1].legend()

plt.show()

## Next Steps

1. `p_raw` 完成后，记录输出目录和配置指纹。
2. 重启 kernel，释放 JAX GPU 显存。
3. 从顶部重新运行，把 `VARIANT` 改为 `p_rms`。
4. 不要复制或复用 `p_raw` 的 normalizer、FE 参数或优化器状态。
5. 两个 FE 都完成后，再分别进行 OL 训练。

如果 Notebook 或服务器中断，只需重新运行 Setup、设备检查和配置单元，然后再次运行训练单元。必须保持相同的 `VARIANT`、`SEED`、输出路径和配置，并继续使用 `resume=True`。